## NOTE: Notebook Settings Variables
SET TO FALSE FOR REAL RUNS

Setting to False keeps full dataset. Setting to True uses very small sample to test functionality.

In [ ]:
TESTING = False

SET TO TRUE FOR FULL NOTEBOOK RUN
Setting to true runs full training. Setting to false skips training, loads model from HF and only runs inference.

In [ ]:
DO_TRAINING = False

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

#import numpy as np # linear algebra
#import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1 Setup

**NYU Deep Learning Spring 2026 — Text-to-SVG Kaggle Competition**

Train a Qwen 2.5-3B model with QLoRA to generate SVGs from text prompts,
then run inference on the test set and produce `submission.csv`.

Built from the starter notebook scaffold, with improvements.

## 1.1 Install dependencies

In [ ]:
# Unsloth must be installed before any other ML imports.
# Restart runtime after this cell if running for the first time.
# Uncomment whenever in a fresh environment
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml huggingface_hub

In [ ]:
!pip install transformers==4.56.2 # unsloth shape mismatch bug

## 1.2 Imports and seed

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset

os.environ["CUDA_VISIBLE_DEVICES"] = "0" # unsloth is only meant for 1 gpu, not kaggle's multiple

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : NVIDIA A100-SXM4-40GB
VRAM     : 42.4 GB


## 1.3 Constants and configuration

All tunable values in one place for easy maintenance.

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────

# System prompt: same as Notebook 01 (used during data filtering)
SYSTEM_PROMPT = (
    "You are an SVG code generator. "
    "Given a visual description, output only valid SVG code. "
    "Use semantic SVG elements: <circle> for circles, <rect> for rectangles, "
    "<ellipse> for ovals, <line> for straight lines, <polygon> for angular shapes, "
    "<path> for curves and complex outlines. "
    "Build complex objects by composing multiple simple shapes. "
    "Use white background unless specified otherwise. "
    "Output SVG code only — no explanation, no markdown, no comments."
)

# Tags that are explicitly allowed
# svg, g, path, rect, circle, ellipse,
# line, polyline, polygon, defs, use,
# symbol, clipPath, mask, linearGradient,
# radialGradient, stop, text, tspan, title,
# desc, style, pattern, marker, filter
ALLOWED_TAGS = [
    "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "linearGradient", "mask",
    "radialGradient", "stop", "text", "tspan", "title",
    "desc", "style", "pattern", "marker", "filter"
]

# Tags that cause a zero score if present in submission
# scripts, event handlers, animation, foreignObject, external references
DISALLOWED_TAGS = [
    "script",
    "animate",
    "animateTransform",
    "animateMotion",
    "foreignObject",
    "set",
]

# Token / sequence limits (from Notebook 01 analysis)
TOKEN_LIMIT    = 2048
MAX_SEQ_LENGTH = TOKEN_LIMIT

# Generation parameters -> shared between generate_svg and generate_svg_batch.
# Edit here once, applies everywhere.
# EDITS for A100: play with token ceiling
MAX_NEW_TOKENS_NORMAL = 768 #TOKEN_LIMIT/2      # normal cap: eos_token stops early, but this is ceiling
MAX_NEW_TOKENS_RETRY  = 1024 #TOKEN_LIMIT   # retry cap:  longer budget for outputs that got cut off
GEN_DEFAULTS = {
    "temperature":          0.5,
    "top_p":                0.9,
    "repetition_penalty":   1.15,  # increase from 1.05 to prevent too much repetition
    "no_repeat_ngram_size": 8,     # forbid any 8-token sequence from repeating
}
# Retry parameters: higher diversity for second attempts
GEN_RETRY = {
    "temperature":          0.9, # increase variance
    "top_p":                0.95,
    "repetition_penalty":   1.3, # increase repetition penalty
    "no_repeat_ngram_size": 8,
}

# Competition constraints
SVG_MAX_CHARS  = 8000   # output SVGs must be ≤ 8,000 characters
SVG_CANVAS     = 256    # competition renders at 256×256 pixels
MAX_PATH_COUNT = 256    # max allowed <path> elements

# Batch size for inference (reduce if any OOM errors!)
INFERENCE_BATCH_SIZE = 64

# Generic output detection threshold
GENERIC_THRESHOLD = 5  # if same path signature appears 5+ times, flag as generic

In [ ]:
# ── Load Env Secret for HuggingFace ────────────────────────────────────────
# should work on both kaggle and colab automatically
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except Exception:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded.")
else:
    print("No HF_TOKEN found.")

HF_TOKEN loaded.


In [ ]:
# ── Training / path configuration ────────────────────────────────────────
# Auto-detects Kaggle vs Colab/other so easy to run in both envs (hopefully!)

# Model
MODEL_ID = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" # use pre-quantized unsloth variant for faster load, same weights
MODEL_ID_NO_UNSLOTH = "Qwen/Qwen2.5-3B-Instruct" # EDITS for A100, inference without unsloth

# HuggingFace repos (used when not on Kaggle)
HF_DATASET_REPO = "aagoluoglu/text-to-svg"
HF_MODEL_REPO   = "aagoluoglu/qwen-svg-lora" #"aagoluoglu/svg-lora-adapter"

# Kaggle paths (used when files exist at these locations)
KAGGLE_TRAIN_PATH   = "/kaggle/input/datasets/ashleygoluoglu/train-filtered-csv/train_filtered.csv"
KAGGLE_TEST_PATH    = "/kaggle/input/competitions/dl-spring-2026-svg-generation-from-text-prompts-extended-deadline/test.csv"
KAGGLE_ADAPTER_PATH = "/kaggle/input/svg-lora-adapter" # if saved as a model in kaggle and connected as input

# Output paths (used when on Kaggle, otherwise uses HF)
ADAPTER_SAVE_DIR = "/kaggle/working/svg-lora-adapter"
CHECKPOINT_DIR   = "/kaggle/working/svg_lora_checkpoints"
SUBMISSION_PATH  = "/kaggle/working/submission.csv"

CONFIG = {
    # Model
    "model_name":       MODEL_ID,
    "max_seq_length":   MAX_SEQ_LENGTH,

    # LoRA adapter settings
    "lora_r":           64,    # og starter: 16, increased to learn SVG syntax
    "lora_alpha":       64,    # alpha=r -> scaling factor of 1.0
    "lora_dropout":     0,     # 0 recommended by Unsloth for speed
    "target_modules":   [
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention projections
        "gate_proj", "up_proj", "down_proj",        # MLP projections
    ],

    # Training
    "num_train_epochs":             3,     # og: 1, increased for better convergence
    "per_device_train_batch_size":  2,     # keep low for safety with r=64 + packing
    "gradient_accumulation_steps":  8,     # effective batch=16
    "learning_rate":                2e-5,  # og: 2e-4, reduced 10x for higher rank LoRA
    "warmup_ratio":                 0.05,
    "weight_decay":                 0.01,
    "lr_scheduler_type":            "cosine",
    "max_grad_norm":                1.0,   # og: 0.3 was too aggressive, clipping too much

    # Logging and checkpoints
    "logging_steps":    20,
    "save_steps":       400,
    "eval_steps":       200,
    "save_total_limit": 2,
    "eval_size":        0.02,

    # Paths
    "output_dir":        CHECKPOINT_DIR,
    "adapter_save_dir":  ADAPTER_SAVE_DIR,

    # Reproducibility
    "seed": SEED,
}

print("CONFIG loaded.")
print(f"  Model              : {CONFIG['model_name']}")
print(f"  max_seq_length     : {CONFIG['max_seq_length']}")
print(f"  LoRA rank          : {CONFIG['lora_r']}")
print(f"  Effective batch    : {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"  Epochs             : {CONFIG['num_train_epochs']}")
print(f"  Learning rate      : {CONFIG['learning_rate']}")

CONFIG loaded.
  Model              : unsloth/Qwen2.5-3B-Instruct-bnb-4bit
  max_seq_length     : 2048
  LoRA rank          : 64
  Effective batch    : 16
  Epochs             : 3
  Learning rate      : 2e-05


# 2 Training

Fine-tune Qwen 2.5-3B with QLoRA on the filtered dataset from Notebook 01.
Uses Unsloth's `FastLanguageModel` for memory-efficient 4-bit training.

## 2.1 Load filtered training data

In [ ]:
if DO_TRAINING:
    # Auto-detect: use Kaggle-attached dataset if available, otherwise pull from HF
    if os.path.exists(KAGGLE_TRAIN_PATH):
        train_raw = Dataset.from_pandas(pd.read_csv(KAGGLE_TRAIN_PATH))
        print(f"Loaded from Kaggle: {KAGGLE_TRAIN_PATH}")
    else:
        train_raw = load_dataset(
            HF_DATASET_REPO,
            data_files={"train": "train_filtered.csv"},
            split="train",
        )
        print(f"Loaded from HuggingFace: {HF_DATASET_REPO}")

    print(f"Examples : {len(train_raw):,}")
    print(f"Columns  : {train_raw.column_names}")

In [ ]:
if DO_TRAINING:
    if TESTING:
        train_raw = train_raw.select(range(10))

## 2.2 Train/eval split

In [ ]:
if DO_TRAINING:
    splits   = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
    train_ds = splits["train"]
    eval_ds  = splits["test"]

    print(f"Train examples : {len(train_ds):,}")
    print(f"Eval examples  : {len(eval_ds):,}")

## 2.3 Load model and attach LoRA

In [ ]:
from unsloth import FastLanguageModel

if DO_TRAINING:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = CONFIG["model_name"],
        max_seq_length = CONFIG["max_seq_length"],
        dtype          = None,       # auto: BF16 if supported, else FP16
        load_in_4bit   = True,       # True: NF4 quantization for base weights
    )

    print(f"Loaded: {CONFIG['model_name']}")
    print(f"Dtype : {model.dtype}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
if DO_TRAINING:
    model = FastLanguageModel.get_peft_model(
        model,
        r                       = CONFIG["lora_r"],
        lora_alpha              = CONFIG["lora_alpha"],
        lora_dropout            = CONFIG["lora_dropout"],
        target_modules          = CONFIG["target_modules"],
        bias                    = "none",
        use_gradient_checkpointing = "unsloth",  # Unsloth's optimized checkpointing
        random_state            = SEED,
    )

    model.print_trainable_parameters()

## 2.4 Format dataset

Each training example is formatted into the Qwen chat template.
`apply_chat_template` inserts the model-specific special tokens
(`<|im_start|>`, `<|im_end|>`) automatically.

In [ ]:
def format_example(example: dict) -> dict:
    """Format a (prompt, svg) pair into the Qwen chat template."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["prompt"]},
        {"role": "assistant", "content": example["svg"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

if DO_TRAINING:
    train_formatted = train_ds.map(format_example, remove_columns=train_ds.column_names)
    eval_formatted  = eval_ds.map(format_example, remove_columns=eval_ds.column_names)

    print(f"Formatted {len(train_formatted):,} train + {len(eval_formatted):,} eval examples")
    print(f"\nSample (first 400 chars):\n{train_formatted[0]['text'][:400]}")

## 2.5 Train

In [ ]:
# Optional: callback to push adapter checkpoints to HuggingFace Hub (only works with internet access)
# hopefully saves things when sessions disconnect
# if no internet (ex: kaggle w/o internet), skip
from transformers import TrainerCallback

class BackupCallback(TrainerCallback):
    """
    - on_save (every save_steps=400): push adapter only to HF (small, fast, but can only use for inference).
    - on_epoch_end: push full checkpoint to HF (larger, slower, but can also use to resume training).
    """
    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        print(f"\n--- Step {step}: pushing adapter ---")
        try:
            model.save_pretrained(CONFIG["adapter_save_dir"])
            tokenizer.save_pretrained(CONFIG["adapter_save_dir"])
            model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN,
                              commit_message=f"adapter step {step}")
            tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
            print(f"  Adapter pushed at step {step}")
        except Exception as e:
            print(f"  Push skipped: {e}")

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        print(f"\n--- Epoch {epoch} complete: pushing full checkpoint ---")
        try:
            # Find the latest checkpoint directory
            ckpt_dirs = sorted(
                [d for d in os.listdir(args.output_dir) if d.startswith("checkpoint-")],
                key=lambda x: int(x.split("-")[1])
            )
            if not ckpt_dirs:
                print("  No checkpoint found, skipping")
                return
            latest_ckpt = os.path.join(args.output_dir, ckpt_dirs[-1])

            from huggingface_hub import HfApi
            api = HfApi(token=HF_TOKEN)
            api.upload_folder(
                folder_path=latest_ckpt,
                repo_id=HF_MODEL_REPO,
                path_in_repo=f"checkpoint-epoch-{epoch}",
                commit_message=f"full checkpoint epoch {epoch}",
            )
            print(f"  Full checkpoint pushed: {latest_ckpt}")
        except Exception as e:
            print(f"  Checkpoint push skipped: {e}")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

if DO_TRAINING:
    training_args = TrainingArguments(
        output_dir                  = CONFIG["output_dir"],
        num_train_epochs            = CONFIG["num_train_epochs"],
        per_device_train_batch_size = CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps = CONFIG["gradient_accumulation_steps"],
        learning_rate               = CONFIG["learning_rate"],
        warmup_ratio                = CONFIG["warmup_ratio"],
        weight_decay                = CONFIG["weight_decay"],
        lr_scheduler_type           = CONFIG["lr_scheduler_type"],
        max_grad_norm               = CONFIG["max_grad_norm"],
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = CONFIG["logging_steps"],
        eval_strategy               = "steps",
        eval_steps                  = CONFIG["eval_steps"],
        save_strategy               = "steps",
        save_steps                  = CONFIG["save_steps"],
        save_total_limit            = CONFIG["save_total_limit"],
        load_best_model_at_end      = True,
        optim                       = "adamw_8bit",
        seed                        = SEED,
        report_to                   = "none",
    )

    trainer = SFTTrainer(
        model              = model,
        tokenizer          = tokenizer,
        train_dataset      = train_formatted,
        eval_dataset       = eval_formatted,
        dataset_text_field = "text",
        max_seq_length     = CONFIG["max_seq_length"],
        packing            = True,  # groups short examples into single sequences
        args               = training_args,
        callbacks          = [BackupCallback()],
    )

    print("Trainer configured.")
    print(f"  Effective batch : {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
    print(f"  Packing         : True")

In [ ]:
if DO_TRAINING:
    if torch.cuda.is_available():
        gpu_mem_before = torch.cuda.memory_allocated() / 1e9
        print(f"GPU memory before training: {gpu_mem_before:.2f} GB")

    t_start      = time.time()
    train_result = trainer.train()
    t_elapsed    = (time.time() - t_start) / 60

    print(f"\nTraining complete.")
    print(f"  Total time   : {t_elapsed:.1f} minutes")
    print(f"  Train loss   : {train_result.training_loss:.4f}")
    print(f"  Steps        : {train_result.global_step}")

    if torch.cuda.is_available():
        gpu_mem_after = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory   : {gpu_mem_after:.2f} GB")

## 2.6 Save adapter

In [ ]:
if DO_TRAINING:
    os.makedirs(CONFIG["adapter_save_dir"], exist_ok=True)

    model.save_pretrained(CONFIG["adapter_save_dir"])
    tokenizer.save_pretrained(CONFIG["adapter_save_dir"])

    # List saved files
    print(f"Saved adapter to: {CONFIG['adapter_save_dir']}")
    total_mb = 0
    for fname in sorted(os.listdir(CONFIG["adapter_save_dir"])):
        fpath = os.path.join(CONFIG["adapter_save_dir"], fname)
        mb    = os.path.getsize(fpath) / 1e6
        total_mb += mb
        print(f"  {fname:<45} {mb:>7.1f} MB")
    print(f"  {'TOTAL':<45} {total_mb:>7.1f} MB")

In [ ]:
# Optional: push adapter to HuggingFace Hub (only works with internet access)
# if no internet (ex: kaggle w/o internet), skip
if DO_TRAINING:
    try:
        model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
        print(f"Pushed to: https://huggingface.co/{HF_MODEL_REPO}")
    except Exception as e:
        print(f"Warning: Push skipped (no internet and/or token?): {e}")

In [ ]:
if DO_TRAINING:
    print("=" * 58)
    print("TRAINING SUMMARY")
    print("=" * 58)
    print(f"  Model              : {CONFIG['model_name']}")
    print(f"  LoRA rank (r)      : {CONFIG['lora_r']}")
    print(f"  LoRA alpha         : {CONFIG['lora_alpha']}")
    print(f"  max_seq_length     : {CONFIG['max_seq_length']}")
    print(f"  Training examples  : {len(train_ds):,}")
    print(f"  Eval examples      : {len(eval_ds):,}")
    print(f"  Epochs             : {CONFIG['num_train_epochs']}")
    print(f"  Effective batch    : {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
    print(f"  Learning rate      : {CONFIG['learning_rate']}")
    print(f"  Packing            : True")
    print(f"  Training steps     : {train_result.global_step}")
    print(f"  Final train loss   : {train_result.training_loss:.4f}")
    print(f"  Training time      : {t_elapsed:.1f} minutes")
    print(f"  Adapter saved to   : {CONFIG['adapter_save_dir']}")
    print("=" * 58)
else:
    print("Training skipped...running inference.")

Training skipped...running inference.


# 3 Inference

Load the trained adapter, generate SVGs for all test prompts,
validate and retry invalid outputs, detect generic outputs, and save submission.

## 3.1 Load adapter for inference

In [ ]:
# EDITS for A100
# unmerged model code block

# Reload model fresh from saved adapter (avoids Unsloth internal state bugs)
# Auto-detect: if just trained, adapter is in ADAPTER_SAVE_DIR.
# On a separate inference-only run, it could be a Kaggle-attached dataset,
# or if not doing training in same place, get from HF
# if not DO_TRAINING:
#     adapter_path = HF_MODEL_REPO
# elif os.path.exists(CONFIG["adapter_save_dir"]):
#     adapter_path = CONFIG["adapter_save_dir"]
# elif os.path.exists(KAGGLE_ADAPTER_PATH):
#     adapter_path = KAGGLE_ADAPTER_PATH
# else:
#     # Fallback: load from HuggingFace Hub
#     adapter_path = HF_MODEL_REPO

# print(f"Loading adapter from: {adapter_path}")

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name     = adapter_path,
#     max_seq_length = MAX_SEQ_LENGTH,
#     dtype          = None,
#     load_in_4bit   = True,
#     token          = HF_TOKEN,
# )
# FastLanguageModel.for_inference(model)
# tokenizer.padding_side = "left"

# print("Model loaded and set to inference mode.")
# if torch.cuda.is_available():
#     print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# EDITS for A100
# merged model code block

# Reload model fresh, using FULL PRECISION for fast inference on A100
# (4-bit quantization is slow, this version for A100 has 40GB VRAM so no need for it)

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Determine adapter path
if not DO_TRAINING:
    adapter_path = HF_MODEL_REPO  # "aagoluoglu/qwen-svg-lora"
elif os.path.exists(CONFIG["adapter_save_dir"]):
    adapter_path = CONFIG["adapter_save_dir"]
elif os.path.exists(KAGGLE_ADAPTER_PATH):
    adapter_path = KAGGLE_ADAPTER_PATH
else:
    adapter_path = HF_MODEL_REPO

print(f"Loading adapter from: {adapter_path}")

# Load base model in bf16 (not quantized, much faster on A100)
# Load without Unsloth, save time on inference
# NOTE: for this method to work and not have issues, need to restart session without any unsloth
# otherwise unsloth has some buggy things and conflicts
# on kaggle, have to use the unmerged version with unsloth
# bc unsloth is used in training and the save & run all format means its always still *there*
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    token=HF_TOKEN,
)

# Load and merge LoRA adapter
model = PeftModel.from_pretrained(base_model, adapter_path, token=HF_TOKEN)
model = model.merge_and_unload()  # Merge LoRA into base for faster inference
model.eval()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct", token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Model loaded and merged for fast inference.")
print(f"Model dtype: {next(model.parameters()).dtype}")
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading adapter from: aagoluoglu/qwen-svg-lora


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded and merged for fast inference.
Model dtype: torch.bfloat16
GPU memory: 6.18 GB


## 3.2 SVG utility functions

In [ ]:
# EDITS for A100
# for sake of inference speed, this function is edited to fix truncated svgs by adding closing tag
# rather than retrying their inference
def extract_svg(text: str) -> str:
    """Extract SVG from decoded model output, handling truncation."""
    # First try to find complete SVG
    match = re.search(r"<svg[\s\S]*?</svg>", text, flags=re.IGNORECASE)
    if match:
        return match.group(0).strip()

    # If no complete SVG, try to salvage truncated one by adding closing tag
    match = re.search(r"<svg[\s\S]*", text, flags=re.IGNORECASE)
    if match:
        truncated = match.group(0).strip()
        # Try to close any open tags and add </svg>
        # Count unclosed path/rect/circle/etc tags
        if "<svg" in truncated.lower() and "</svg>" not in truncated.lower():
            # Close the SVG tag
            # Remove any incomplete attribute at the end (ends mid-quote or mid-tag)
            truncated = re.sub(r'\s+\w+="[^"]*$', '', truncated)  # Remove incomplete attribute
            truncated = re.sub(r'<[^>]*$', '', truncated)  # Remove incomplete tag
            truncated += "</svg>"
            return truncated

    return ""


def is_valid_svg(svg_str: str) -> bool:
    """Check if string is non-empty, parses as valid XML, and has <svg> root."""
    if not svg_str or not svg_str.strip():
        return False
    try:
        root = ET.fromstring(svg_str)
        return root.tag.lower().endswith("svg")
    except ET.ParseError:
        return False


def sanitize_svg(svg_str: str) -> str:
    """Remove disallowed elements (animate, script, hallucinations, etc.) that cause zero score."""
    # Find all tag names in the SVG
    all_tags = set(re.findall(r'</?(\w+)[\s/>]', svg_str, flags=re.IGNORECASE))

    # Remove anything not allowed + anything explicitly disallowed (in case somehow overlap, and covers hallucinated tags)
    bad_tags = (all_tags - set(ALLOWED_TAGS) - {"svg"}) | set(DISALLOWED_TAGS)

    for tag in bad_tags:
        svg_str = re.sub(
            rf'<{tag}[\s\S]*?</{tag}>', '', svg_str, flags=re.IGNORECASE
        )
        svg_str = re.sub(
            rf'<{tag}[^>]*/>', '', svg_str, flags=re.IGNORECASE
        )

    return svg_str


def ensure_dimensions(svg_str: str) -> str:
    """Ensure the root <svg> has width="256" height="256" for competition rendering.

    Preserves the model's original viewBox (the renderer scales internal
    coordinates to fit the pixel canvas). Only adds a default viewBox
    if one is missing entirely.
    """
    if not svg_str:
        return svg_str

    # Set width and height to competition canvas size
    svg_str = re.sub(
        r'\bwidth="[^"]*"',
        f'width="{SVG_CANVAS}"',
        svg_str, count=1
    )
    svg_str = re.sub(
        r'\bheight="[^"]*"',
        f'height="{SVG_CANVAS}"',
        svg_str, count=1
    )

    # Add width/height if not present at all
    if 'width=' not in svg_str:
        svg_str = svg_str.replace(
            '<svg', f'<svg width="{SVG_CANVAS}"', 1
        )
    if 'height=' not in svg_str:
        svg_str = svg_str.replace(
            '<svg', f'<svg height="{SVG_CANVAS}"', 1
        )

    # Add viewBox if completely missing (default to 256x256)
    if 'viewBox' not in svg_str and 'viewbox' not in svg_str:
        svg_str = svg_str.replace(
            '<svg',
            f'<svg viewBox="0 0 {SVG_CANVAS} {SVG_CANVAS}"',
            1
        )

    return svg_str


def enforce_char_limit(svg_str: str, limit: int = SVG_MAX_CHARS) -> str:
    """Return empty string if SVG exceeds competition character limit.

    Over-limit SVGs score zero, so it's better to fall back to a simple
    valid SVG than to submit one that will be rejected.
    """
    if len(svg_str) > limit:
        return ""
    return svg_str


def post_process_svg(decoded_text: str) -> str:
    """Full post-processing pipeline for raw model output.

    Pipeline: extract -> ensure dimensions -> enforce char limit.
    Sanitization is applied separately so we can track its effect.
    """
    svg = extract_svg(decoded_text)
    svg = ensure_dimensions(svg)
    svg = enforce_char_limit(svg)
    return svg


def fallback_svg(prompt: str) -> str:
    """Minimal valid SVG as last resort. Scores low but not zero."""
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" '
        f'viewBox="0 0 {SVG_CANVAS} {SVG_CANVAS}" '
        f'width="{SVG_CANVAS}" height="{SVG_CANVAS}">'
        f'<rect width="{SVG_CANVAS}" height="{SVG_CANVAS}" fill="white"/>'
        '</svg>'
    )


def get_path_signature(svg_str: str) -> str:
    """Extract first 60 chars of the first d= attribute as a fingerprint.

    Used to detect generic/memorized outputs: if many different prompts
    produce the same path signature, the model is likely outputting a
    memorized default rather than prompt-specific content.
    """
    match = re.search(r'd="([^"]{0,60})', svg_str)
    return match.group(1) if match else ""


print("SVG utility functions loaded.")

SVG utility functions loaded.


## 3.3 Generation functions

In [ ]:
def _build_gen_kwargs(gen_params: dict, max_new_tokens: int) -> dict:
    """Build kwargs dict for model.generate().

    Centralizes generation parameters so generate_svg and generate_svg_batch
    share the exact same settings. Edit GEN_DEFAULTS or GEN_RETRY at the top
    of the notebook -> both functions pick up the changes automatically!
    """
    return {
        "max_new_tokens":       max_new_tokens,
        "do_sample":            True,
        "use_cache":            True,
        "temperature":          gen_params["temperature"],
        "top_p":                gen_params["top_p"],
        "repetition_penalty":   gen_params["repetition_penalty"],
        "no_repeat_ngram_size": gen_params["no_repeat_ngram_size"],
        "eos_token_id":         tokenizer.eos_token_id,
        "pad_token_id":         tokenizer.pad_token_id,
    }


def generate_svg(
    prompt: str,
    gen_params: dict = None,
    max_new_tokens: int = None,
) -> str:
    """Generate an SVG string from a single text prompt."""
    if gen_params is None:
        gen_params = GEN_DEFAULTS
    if max_new_tokens is None:
        max_new_tokens = MAX_NEW_TOKENS_NORMAL

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, **_build_gen_kwargs(gen_params, max_new_tokens)
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    decoded    = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return post_process_svg(decoded)

# EDITS for A100
# editing this function slightly to try to get faster results
@torch.inference_mode()  # Faster than torch.no_grad()
def generate_svg_batch(prompts: list[str], gen_params: dict = None, max_new_tokens: int = None) -> list[str]:
    gen_params = gen_params or GEN_DEFAULTS
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS_NORMAL

    messages_batch = [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": p},
        ]
        for p in prompts
    ]

    input_texts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_batch
    ]

    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        **gen_params,
    )

    # Decode only new tokens
    svgs = []
    for i, output in enumerate(output_ids):
        new_tokens = output[inputs["input_ids"].shape[1]:]
        decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
        svgs.append(extract_svg(decoded))

    return svgs


print("Generation functions loaded.")

Generation functions loaded.


## 3.4 Quick check

Generate a few test prompts to visually confirm the model is working
before committing to the full 1,000-prompt inference loop.

In [ ]:
from IPython.display import SVG, HTML, display

if TESTING:
    quick_check_prompts = [
        "a simple red circle on a white background",
        "a blue five-pointed star",
        "a black cat silhouette",
        "a green tree with a brown trunk",
        "a yellow sun with rays",
        "a simple blue bird icon",
    ]

    print("Quick check: generating SVGs for 6 test prompts...\n")
    html_parts = []
    for prompt in quick_check_prompts:
        svg   = generate_svg(prompt)
        svg   = sanitize_svg(svg) if svg else ""
        valid = is_valid_svg(svg)

        print(f"Prompt : {prompt}")
        print(f"  Valid  : {valid}")
        print(f"  Length : {len(svg)} chars")

        if valid:
            html_parts.append(
                f"<div style='display:inline-block; margin:10px; "
                f"vertical-align:top; width:180px;'>"
                f"<p style='font-size:10px; color:#555; "
                f"margin:4px 0;'>{prompt}</p>"
                f"{svg}</div>"
            )
        print()

    if html_parts:
        display(HTML("".join(html_parts)))
    else:
        print("WARNING: No valid SVGs generated. Check model loading and adapter.")

## 3.5 Load test prompts

In [ ]:
# Auto-detect test prompts location
if os.path.exists(KAGGLE_TEST_PATH):
    test_df = pd.read_csv(KAGGLE_TEST_PATH)
    print(f"Loaded from Kaggle: {KAGGLE_TEST_PATH}")
else:
    ds = load_dataset(
        HF_DATASET_REPO, data_files={"test": "test.csv"}, split="test"
    )
    test_df = ds.to_pandas()
    print(f"Loaded from HuggingFace: {HF_DATASET_REPO}")

print(f"Test prompts: {len(test_df):,} rows")
print(f"Columns     : {list(test_df.columns)}")
print(f"\nSample prompts:")
for p in test_df["prompt"].sample(3, random_state=42).values:
    print(f"  - {p[:100]}")

Loaded from HuggingFace: aagoluoglu/text-to-svg
Test prompts: 1,000 rows
Columns     : ['id', 'prompt']

Sample prompts:
  - The image shows a black icon of a stylized pencil with a pointed tip inside a black square border on
  - An image features a black rectangular frame surrounding a white square, which contains two black cir
  - A speaker icon with concentric circles and corners marked by black dots, representing stereo audio e


## 3.6 Full inference loop

Generate SVGs for all test prompts. Deduplicate prompts first
to avoid redundant generation. Retry invalid outputs with higher
temperature and longer token budget before falling back.

In [ ]:
if TESTING:
    test_df = test_df.sample(3, random_state=42)

In [ ]:
from tqdm import tqdm
import time

unique_prompts = list(dict.fromkeys(test_df["prompt"].tolist()))
prompt_counts  = Counter(test_df["prompt"].tolist())
dup_prompts    = {p for p, c in prompt_counts.items() if c > 1}
print(f"Total prompts  : {len(test_df):,}")
print(f"Unique prompts : {len(unique_prompts):,}")
print(f"Duplicates     : {len(dup_prompts)}")
print(f"\nGenerating in batches of {INFERENCE_BATCH_SIZE}...")

# EDITS for A100
# do a warmup generation separately for time counting
print("Warming up model (first inference compiles CUDA kernels)...")
warmup_start = time.time()
_ = generate_svg("a red circle")
print(f"Warm-up done in {time.time() - warmup_start:.1f}s\n")

results = {}
fallback_ids = []
retry_ids = []

# calc & print num total batches before starting
total_batches = (len(unique_prompts) + INFERENCE_BATCH_SIZE - 1) // INFERENCE_BATCH_SIZE
print(f"Total batches: {total_batches}")

t_start = time.time()

# index batches for reporting, helps understand time it will take
for batch_idx, batch_start in enumerate(range(0, len(unique_prompts), INFERENCE_BATCH_SIZE)):
    batch_prompts = unique_prompts[batch_start : batch_start + INFERENCE_BATCH_SIZE]

    # print update at start of batch
    print(f"\nBatch {batch_idx + 1}/{total_batches} ({len(batch_prompts)} prompts)...", flush=True)

    batch_time = time.time()
    svgs = generate_svg_batch(batch_prompts) # should need fewer retries, as new extract function adds closing tags to truncated outputs
    batch_duration = time.time() - batch_time

    print(f"  Done in {batch_duration:.1f}s", flush=True)

    for prompt, svg in zip(batch_prompts, svgs):
        svg = sanitize_svg(svg) if svg else ""

        # Retry individually if invalid -> higher temp + longer token budget
        if not is_valid_svg(svg):
            retry_ids.append(prompt[:40])
            svg = generate_svg(prompt, gen_params=GEN_RETRY, max_new_tokens=MAX_NEW_TOKENS_RETRY)
            svg = sanitize_svg(svg) if svg else ""

        # Final fallback if still invalid
        if not is_valid_svg(svg):
            fallback_ids.append(prompt[:40])
            svg = fallback_svg(prompt)

        results[prompt] = svg

    # progress logging
    valid_count = sum(1 for p in batch_prompts if is_valid_svg(results[p]))
    print(f"  Valid: {valid_count}/{len(batch_prompts)}, Retries so far: {len(retry_ids)}, Fallbacks: {len(fallback_ids)}", flush=True)

t_total = (time.time() - t_start) / 60
print(f"\nDone! {t_total:.1f} minutes total")
print(f"Retries: {len(retry_ids)}, Fallbacks: {len(fallback_ids)}")

Total prompts  : 1,000
Unique prompts : 977
Duplicates     : 3

Generating in batches of 64...
Warming up model (first inference compiles CUDA kernels)...
Warm-up done in 25.2s

Total batches: 16

Batch 1/16 (64 prompts)...
  Done in 109.5s
  Valid: 64/64, Retries so far: 20, Fallbacks: 9

Batch 2/16 (64 prompts)...
  Done in 106.8s
  Valid: 64/64, Retries so far: 35, Fallbacks: 16

Batch 3/16 (64 prompts)...
  Done in 105.7s
  Valid: 64/64, Retries so far: 48, Fallbacks: 25

Batch 4/16 (64 prompts)...
  Done in 106.2s
  Valid: 64/64, Retries so far: 61, Fallbacks: 32

Batch 5/16 (64 prompts)...
  Done in 105.9s
  Valid: 64/64, Retries so far: 85, Fallbacks: 47

Batch 6/16 (64 prompts)...
  Done in 106.2s
  Valid: 64/64, Retries so far: 112, Fallbacks: 55

Batch 7/16 (64 prompts)...
  Done in 107.6s
  Valid: 64/64, Retries so far: 130, Fallbacks: 62

Batch 8/16 (64 prompts)...
  Done in 106.5s
  Valid: 64/64, Retries so far: 143, Fallbacks: 67

Batch 9/16 (64 prompts)...
  Done in 109.

## 3.7 Generic output detection and retry

If the model generates the same path data for many different prompts,
those are likely memorized defaults rather than prompt-specific outputs.
Retry with higher temperature and repetition penalty.

In [ ]:
sig_counts    = Counter()
prompt_by_sig = {}

for prompt, svg in results.items():
    sig = get_path_signature(svg)
    if sig:
        sig_counts[sig] += 1
        prompt_by_sig.setdefault(sig, []).append(prompt)

generic_sigs    = {s for s, c in sig_counts.items() if c >= GENERIC_THRESHOLD}
generic_prompts = set()
for sig in generic_sigs:
    generic_prompts.update(prompt_by_sig[sig])

print(f"Unique path signatures : {len(sig_counts)}")
print(f"Generic signatures (>={GENERIC_THRESHOLD}x): {len(generic_sigs)}")
print(f"Prompts to retry       : {len(generic_prompts)}")
if generic_sigs:
    print("\nMost repeated signatures:")
    for sig, count in sig_counts.most_common(5):
        print(f"  {count}x : {sig[:50]}...")

if not generic_sigs:
    print("No generic outputs detected. All outputs look unique.")
else:
    print(f"\nRetrying {len(generic_prompts)} generic outputs...\n")
    retry_generic_count = 0

    for prompt in generic_prompts:
        svg = generate_svg(
            prompt,
            gen_params=GEN_RETRY,
            max_new_tokens=MAX_NEW_TOKENS_RETRY,
        )
        svg = sanitize_svg(svg) if svg else ""

        if is_valid_svg(svg):
            results[prompt] = svg
            retry_generic_count += 1
        # if invalid, keep original rather than overwriting

    print(f"Successfully replaced {retry_generic_count}/{len(generic_prompts)} generic outputs")

Unique path signatures : 534
Generic signatures (>=5x): 1
Prompts to retry       : 9

Most repeated signatures:
  9x : M0 0h24v24H0z...
  2x : gradient...
  2x : m0 0h24v24h-24z...
  2x : true...
  1x : M-150,-3 L-103 10 C-114 -.9-.51 15 .9l9 11c-.83 0-...

Retrying 9 generic outputs...

Successfully replaced 4/9 generic outputs


## 3.8 Build and validate submission

In [ ]:
# Map results back to original row order
rows = []
for _, row in test_df.iterrows():
    svg = results.get(row["prompt"], None)
    if not svg or not is_valid_svg(svg):
        svg = fallback_svg(row["prompt"])
    rows.append({"id": row["id"], "svg": svg})

submission_df = pd.DataFrame(rows)

print(f"Submission rows : {len(submission_df):,}")
print(f"Null SVGs       : {submission_df['svg'].isnull().sum()}")
print(f"Empty SVGs      : {(submission_df['svg'] == '').sum()}")

Submission rows : 1,000
Null SVGs       : 0
Empty SVGs      : 0


In [ ]:
# Final validity check
submission_df["valid"]   = submission_df["svg"].apply(is_valid_svg)
submission_df["svg_len"] = submission_df["svg"].str.len()

n_valid   = submission_df["valid"].sum()
n_invalid = (~submission_df["valid"]).sum()

print(f"Valid SVGs   : {n_valid:,} ({100*n_valid/len(submission_df):.1f}%)")
print(f"Invalid SVGs : {n_invalid:,} ({100*n_invalid/len(submission_df):.1f}%)")
print(f"\nSVG length distribution:")
for pct in [25, 50, 75, 90, 100]:
    print(f"  p{pct:3d}: {submission_df['svg_len'].quantile(pct/100):.0f} chars")
print(f"\nOver {SVG_MAX_CHARS:,} chars: {(submission_df['svg_len'] > SVG_MAX_CHARS).sum()}")

if n_invalid > 0:
    print("\nInvalid rows (will score zero):")
    print(submission_df[~submission_df["valid"]][["id", "svg"]].head(5))
else:
    print("\nAll outputs valid!")

In [ ]:
# Visual quick check: render 3 random outputs
from IPython.display import HTML, display

quick = submission_df.merge(
    test_df[["id", "prompt"]], on="id"
).sample(n=3, random_state=42).reset_index(drop=True)

html_parts = []
for _, row in quick.iterrows():
    html_parts.append(
        f"<div style='display:inline-block; margin:10px; "
        f"vertical-align:top; width:200px;'>"
        f"<p style='font-size:10px; color:#555; "
        f"margin:4px 0;'>{row['prompt'][:80]}</p>"
        f"{row['svg']}</div>"
    )
display(HTML("".join(html_parts)))

## 3.9 Save submission

In [ ]:
import os

# EDITS for A100
# save to colab content directory instead
# note: run summary printed saved to old kaggle path bc was not rerun after locally saving again to colab
# to preserve accurate training & inference time stamps
SUBMISSION_PATH = "./submission.csv"

final_submission = submission_df[["id", "svg"]]
final_submission.to_csv(SUBMISSION_PATH, index=False)

print(f"Saved : {os.path.abspath(SUBMISSION_PATH)}")
print(f"Rows  : {len(final_submission):,}")
print(f"Size  : {os.path.getsize(SUBMISSION_PATH) / 1e6:.2f} MB")

Saved : /content/submission.csv
Rows  : 1,000
Size  : 0.53 MB


In [ ]:
# Optional: upload submission to huggingface
try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=SUBMISSION_PATH,
        path_in_repo="./submission.csv",
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        commit_message="submission from 02_train_and_inference",
    )
    print(f"Submission uploaded to: https://huggingface.co/datasets/{HF_DATASET_REPO}")
except Exception as e:
    print(f"Upload skipped (no internet and/or token?): {e}")

Submission uploaded to: https://huggingface.co/datasets/aagoluoglu/text-to-svg


## 3.10 Run summary

In [ ]:
print("=" * 55)
print("INFERENCE SUMMARY")
print("=" * 55)
print(f"  Adapter              : {adapter_path}")
print(f"  Base model           : {MODEL_ID}")
print(f"  Test prompts         : {len(test_df):,}")
print(f"  Unique prompts       : {len(unique_prompts):,}")
print(f"  Duplicate prompts    : {len(dup_prompts)}")
print(f"  Retries (invalid)    : {len(retry_ids)}")
print(f"  Fallbacks used       : {len(fallback_ids)}")
print(f"  Generic retries      : {len(generic_prompts)}")
print(f"  Valid in submission  : {n_valid:,} ({100*n_valid/len(submission_df):.1f}%)")
print(f"  Total runtime        : {t_total:.1f} minutes")
print(f"  Output file          : {SUBMISSION_PATH}")
print("=" * 55)

INFERENCE SUMMARY
  Adapter              : aagoluoglu/qwen-svg-lora
  Base model           : unsloth/Qwen2.5-3B-Instruct-bnb-4bit
  Test prompts         : 1,000
  Unique prompts       : 977
  Duplicate prompts    : 3
  Retries (invalid)    : 295
  Fallbacks used       : 158
  Generic retries      : 9
  Valid in submission  : 1,000 (100.0%)
  Total runtime        : 181.6 minutes
  Output file          : /kaggle/working/submission.csv
